### ver2의 sample분석(MBPP)_prompt_v1
기존 프롬프트의 문제점 파악.   
ISSUE : 기존 입력 프롬프트 강화

In [ ]:
# 샘플 하나를 가져와서 자세히 분석하기.   
import json
from collections import Counter
import statistics
import pandas as pd

# 경로를 여기서 직접 지정
path = "../../results/phase1_ver2/sample/single_mbpp/details.jsonl"

def load_results(path):
    data = []
    with open(path, "r") as f:
        for line in f:
            data.append(json.loads(line))
    return data

data = load_results(path)
len(data)

df = pd.DataFrame(data)
df.head(3)

,dataset,task_id,method,model_name,attempt_idx,prompt,raw_output,generated_code,status,passed,exec_success,error_type,error_message,latency_sec,meta,timeout
0,mbpp,601,single,Qwen/Qwen2.5-Coder-7B-Instruct,0,Write Python code to solve the following probl...,"A pair (a, b) can follow another pair (c, d) i...","A pair (a, b) can follow another pair (c, d) i...",fail,False,False,syntax_error,"Traceback (most recent call last):\n File ""/h...",2.466406,"{'code_exec_passed': False, 'setup_exec_passed...",False
1,mbpp,602,single,Qwen/Qwen2.5-Coder-7B-Instruct,0,Write Python code to solve the following probl...,"If no character is repeated, return -1.\n```py...","If no character is repeated, return -1.\n\ndef...",fail,False,False,syntax_error,"Traceback (most recent call last):\n File ""/h...",1.023708,"{'code_exec_passed': False, 'setup_exec_passed...",False
2,mbpp,603,single,Qwen/Qwen2.5-Coder-7B-Instruct,0,Write Python code to solve the following probl...,A lucid number is defined as a number that can...,A lucid number is defined as a number that can...,fail,False,False,syntax_error,"Traceback (most recent call last):\n File ""/h...",2.053296,"{'code_exec_passed': False, 'setup_exec_passed...",False


In [2]:
# 전체 요약
total = len(data)
passed = sum(1 for x in data if x["status"] == "pass")
timeout = sum(1 for x in data if x["status"] == "timeout")
failed = total - passed - timeout

print("=" * 60)
print(f"📂 File: {path}")
print("📊 Overall")
print(f"Total: {total}")
print(f"Pass: {passed}")
print(f"Fail: {failed}")
print(f"Timeout: {timeout}")
print(f"Pass@1: {passed / total:.4f}")

📂 File: ../../results/phase1_ver2/sample/single_mbpp/details.jsonl
📊 Overall
Total: 5
Pass: 0
Fail: 5
Timeout: 0
Pass@1: 0.0000


In [3]:
# error distribution
error_counter = Counter(
    x["error_type"] for x in data if x["error_type"] is not None
)

print("\n📉 Error Type Distribution")
for k, v in error_counter.most_common():
    print(f"{k}: {v}")
    
latencies = [x["latency_sec"] for x in data if x["latency_sec"] is not None]

if latencies:
    print("\n⏱ Latency")
    print(f"Avg: {statistics.mean(latencies):.3f}s")
    print(f"Max: {max(latencies):.3f}s")


📉 Error Type Distribution
syntax_error: 5

⏱ Latency
Avg: 1.977s
Max: 3.102s


### 실패한 샘플의 generated_code(모델이 생성한 str) 확인

In [11]:
failures = [x for x in data if x["status"] == "fail"][:1]

for f in failures:
    print("=" * 80)
    print(f"🧪 TASK: {f['task_id']}")
    print(f"🧪 ERROR: {f['error_type']}")
    print("\n============= RAW INPUT (head) =============")
    print(f["prompt"])
    print("\n============= RAW OUTPUT (head) =============")
    print(f["raw_output"])
    print("\n============= GENERATED CODE (head) =============")
    print(f["generated_code"])
    print("\n========================================================")

🧪 TASK: 601
🧪 ERROR: syntax_error

============= RAW INPUT (head) =============
Write Python code to solve the following problem.
Return only code.
Your code must include all necessary definitions, such as helper functions or classes.

Problem:
Write a function to find the longest chain which can be formed from the given set of pairs.


============= RAW OUTPUT (head) =============
A pair (a, b) can follow another pair (c, d) if b < c. The function should return the length of the longest chain that can be formed.

```python
def findLongestChain(pairs):
    # Your code here
```

Example:
Input: pairs = [[1, 2], [2, 3], [3, 4]]
Output: 3

Explanation: The longest chain that can be formed is [(1, 2), (2, 3), (3, 4)], so the function should return 3.

Note:
- You can assume that the input list of pairs will not be empty.
- Each pair in the input list will have exactly two integers.
- The order of the pairs in the input list does not matter.
:



============= GENERATED CODE (head) ========

입력하는 프롬프트를 좀 더 강조함

In [ ]:
# 좀 더 단순한 기존 ver0
# def build_mbpp_prompt(sample: MBPPSample) -> str:
#     """
#     MBPP는 자연어 문제 설명을 바탕으로
#     실행 가능한 전체 Python 코드를 생성하도록 유도한다.

#     중요:
#     - 함수만이 아니라 필요한 helper class / helper function까지
#       모두 포함한 '전체 코드'를 생성해야 할 수 있다.
#     - 따라서 prompt에서 'all necessary definitions'를 명시한다.
#     """
#     return (
#         "Write Python code to solve the following problem.\n"
#         "Return only code.\n"
#         "Your code must include all necessary definitions, "
#         "such as helper functions or classes.\n\n"
#         f"Problem:\n{sample.problem_text}\n"
#     )


In [ ]:
# ver2
# def build_mbpp_prompt(sample: MBPPSample) -> str:
#     """
#     MBPP는 자연어 문제 설명을 바탕으로
#     실행 가능한 전체 Python 코드를 생성하도록 유도한다.

#     중요:
#     - 설명 금지
#     - markdown fence 금지
#     - placeholder 금지
#     - 필요한 helper class / helper function 포함
#     """
#     return (
#         "You are a Python programmer.\n"
#         "Write a complete Python solution for the following problem.\n"
#         "Output only executable Python code.\n"
#         "Do not include any explanation, markdown fences, examples, or comments like 'Your code here'.\n"
#         "Your code must include all necessary definitions such as helper functions or classes.\n"
#         "Do not leave the function body empty.\n\n"
#         f"Problem:\n{sample.problem_text}\n"
#     )